# JE Validator server on Kaggle

Runs the JE Validator web page on Kaggle's free GPUs, so you can open it in your browser and upload files there:
PaddleOCR (PP-OCRv6, and PaddleOCR-VL-1.6 if you set `OCR_MODE` to hybrid, auto or vl) on one GPU, and an Ollama
model for validation on the other.

**Before you run** (right-hand panel, *Settings*):
1. **Accelerator: GPU T4 x2.** The P100 is too old for PaddlePaddle, which needs compute capability 7.0 or higher.
2. **Internet: On.** This needs a phone-verified Kaggle account.
3. **Add Input:** the private dataset with the code (`je-validator-code`), or set `GIT_URL` below instead.

Then click **Run All**. The first start takes about 10-15 minutes: installing packages and downloading the models.
The last cell prints the page's address and password.

**Use sample or made-up files only.** Kaggle and the tunnel (Cloudflare) are outside Genpact; don't upload real JEs
or client documents. The page's address is public but random, and the password keeps others out.

In [ ]:
# --- Settings -------------------------------------------------------------------------
CODE_DATASET = "je-validator-code"   # the project, uploaded as a private dataset...
GIT_URL = ""                         # ...or clone it instead, e.g. "https://github.com/<you>/JE_Extractor.git"
GIT_BRANCH = "JE_Extractor_OCR"

LOCAL_MODEL = "qwen2.5:7b"               # first reviewer
LOCAL_ESCALATION_MODEL = "qwen2.5:14b"   # second reviewer when confidence is low; "" for none
CONFIDENCE_THRESHOLD = 80                # 0-100; below this, escalate, then human review
OCR_MODE = "ppocr"                       # ppocr (most accurate in the gold-set evaluation) | hybrid | auto | vl
OCR_MODEL_SIZE = "medium"                # PP-OCRv6 size: tiny | small | medium (most accurate; fine on a GPU)
LOCAL_NUM_CTX = 32768                    # model context; raise for long documents
APP_PASSWORD = ""                        # the page's password; "" makes up a new one each run

## 1. Check the GPUs

In [ ]:
import os, subprocess

gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip().splitlines()
print("\n".join(gpus) or "No GPU")
assert gpus, "Turn on a GPU: Settings -> Accelerator -> GPU T4 x2"
assert all(float(g.split(",")[1]) >= 7.0 for g in gpus), \
    "PaddlePaddle needs compute capability 7.0+: choose GPU T4 x2, not P100"

# OCR on the first GPU, Ollama on the second (or both on the only one).
OCR_GPU, OLLAMA_GPU = "0", "1" if len(gpus) > 1 else "0"

## 2. Get the code

In [ ]:
import shutil, zipfile
from pathlib import Path

APP = Path("/kaggle/working/JE_Extractor")
if APP.exists():
    shutil.rmtree(APP)
if GIT_URL:
    subprocess.run(["git", "clone", "-q", "-b", GIT_BRANCH, GIT_URL, str(APP)], check=True)
else:
    # the dataset's files, or the zip itself if Kaggle didn't unzip it (kaggle/eval/push.ps1 code uploads a zip)
    found = [p.parent.parent for p in Path("/kaggle/input").rglob("je_validator/__main__.py") if CODE_DATASET in p.parts]
    archives = [p for p in Path("/kaggle/input").rglob("*.zip") if CODE_DATASET in p.parts]
    assert found or archives, f"Couldn't find the code: add the '{CODE_DATASET}' dataset as input, or set GIT_URL"
    if found:
        shutil.copytree(found[0], APP)
    else:
        zipfile.ZipFile(archives[0]).extractall(APP)
print("Code:", APP, sorted(p.name for p in APP.iterdir()))

## 3. Install the Python packages
The same hash-pinned versions as on the laptop (the lock files), in a Python 3.12 venv outside the notebook's own
packages: PaddlePaddle for CUDA 12.6, then the app, PP-OCRv6 and PaddleOCR-VL-1.6 (without OpenAI's packages).

In [ ]:
VENV = Path("/tmp/je-venv")  # outside /kaggle/working, so it isn't saved as output
PY = str(VENV / "bin" / "python")
!pip install -q uv
!python -m uv venv -q --seed -p 3.12 {VENV}
!{PY} -m pip install -q paddlepaddle-gpu==3.2.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!python -m uv pip install -q --python {PY} --require-hashes -r {APP}/requirements.lock -r {APP}/requirements-ocr.lock -r {APP}/requirements-vl.lock
!CUDA_VISIBLE_DEVICES=0 {PY} -c "import paddle; paddle.utils.run_check()"

## 4. Install and start Ollama, then download the models

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import time, urllib.error, urllib.request


def wait_for(url, what, log):
    for _ in range(120):
        try:
            urllib.request.urlopen(url)
            return
        except urllib.error.HTTPError:
            return  # it answered (e.g. 401 for a password): it is up
        except OSError:
            time.sleep(1)
    raise RuntimeError(f"{what} didn't start - see {log}")


ollama_env = {**os.environ, "CUDA_VISIBLE_DEVICES": OLLAMA_GPU,
              "OLLAMA_FLASH_ATTENTION": "1", "OLLAMA_KV_CACHE_TYPE": "q8_0"}  # halves the context's memory
ollama_proc = subprocess.Popen(["ollama", "serve"], env=ollama_env, stdout=open("/kaggle/working/ollama.log", "w"),
                               stderr=subprocess.STDOUT)
wait_for("http://127.0.0.1:11434", "Ollama", "/kaggle/working/ollama.log")

for model in filter(None, [LOCAL_MODEL, LOCAL_ESCALATION_MODEL]):
    print("Downloading", model, "...")
    subprocess.run(["ollama", "pull", model], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

## 5. Start the web page
The app runs on this machine; a Cloudflare tunnel gives it an address you can open from your browser.

In [ ]:
import re, secrets

APP_PASSWORD = APP_PASSWORD or secrets.token_urlsafe(9)
server_env = {
    **os.environ,
    "CUDA_VISIBLE_DEVICES": OCR_GPU,  # PaddleOCR sees only its GPU, as gpu:0
    "OCR_DEVICE": "gpu:0",
    "OCR_MODE": OCR_MODE,
    "OCR_MODEL_SIZE": OCR_MODEL_SIZE,
    "LLM_PROVIDER": "ollama",
    "LLM_MODEL": LOCAL_MODEL,
    "LLM_ESCALATION_MODEL": LOCAL_ESCALATION_MODEL,
    "OLLAMA_NUM_CTX": str(LOCAL_NUM_CTX),
    "CONFIDENCE_THRESHOLD": str(CONFIDENCE_THRESHOLD),
    "TEMPLATES_DB": "/kaggle/working/templates.db",
    "APP_PASSWORD": APP_PASSWORD,
}
# The two sample templates; add your own on the page.
subprocess.run([PY, "-m", "je_validator", "templates", "add", *map(str, sorted((APP / "sample_templates").glob("*.json")))],
               cwd=APP, env=server_env, check=True)

server = subprocess.Popen([PY, "-m", "je_validator", "serve"], cwd=APP, env=server_env,
                          stdout=open("/kaggle/working/server.log", "w"), stderr=subprocess.STDOUT)
wait_for("http://127.0.0.1:8000", "The app", "/kaggle/working/server.log")

!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
tunnel_log = Path("/kaggle/working/tunnel.log")
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=tunnel_log.open("w"), stderr=subprocess.STDOUT)
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel_log.read_text())
    if m:
        break
    time.sleep(1)
else:
    raise RuntimeError(f"The tunnel didn't start - see {tunnel_log}")

from IPython.display import HTML, display
display(HTML(f'<h3>Open <a href="{m.group(0)}" target="_blank">{m.group(0)}</a></h3>'
             f'<p>User name: anything. Password: <code>{APP_PASSWORD}</code></p>'
             f'<p>It can take a minute before the address works.</p>'))

## 6. Keep the server running

This cell keeps the session busy and prints the app's log. Leave this tab open while you use the page.
Stop the cell (■) to shut the server down, then stop the session (power icon, top bar) so it doesn't use up your
weekly GPU hours.

Everything here is lost when the session ends, including templates you added on the page. Keep your
template JSON files on your own machine and upload them again next time.

In [ ]:
printed = 0
try:
    while server.poll() is None:
        log = Path("/kaggle/working/server.log").read_text()
        print(log[printed:], end="")
        printed = len(log)
        time.sleep(10)
    print("The app stopped - see /kaggle/working/server.log")
except KeyboardInterrupt:
    for p in (tunnel, server, ollama_proc):
        p.terminate()
    print("Stopped.")